# Chapter 2 — Strings and Slicing

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 2.

---


### Item 10 — Know the Differences Between `bytes` and `str`

`bytes` holds raw 8-bit values; `str` holds Unicode code points (text). They are never interchangeable — you can't concatenate a `bytes` object with a `str`, and comparing them for equality always returns `False` even if they "look" the same. `.encode()` turns `str` into `bytes`; `.decode()` turns `bytes` into `str` — you must always know which one you currently hold.


In [1]:
text = 'café'
data = text.encode('utf-8')

print(type(text), type(data))
print(text == data)              # always False -- different types, never equal

try:
    combined = text + data       # TypeError -- can't mix str and bytes
except TypeError as e:
    print('TypeError:', e)

# Round-trip: str -> bytes -> str
back_to_text = data.decode('utf-8')
print(back_to_text == text)


<class 'str'> <class 'bytes'>
False
TypeError: can only concatenate str (not "bytes") to str
True


### Item 11 — Prefer Interpolated F-Strings over C-Style Format Strings and `str.format`

Python has three ways to build strings from values: old C-style `%` formatting, `.format()`, and f-strings (`f'{value}'`, 3.6+). F-strings win because the expression sits directly inside the string (nothing to count/match up positionally), they support arbitrary expressions and format specs inline (`f'{value:.2f}'`), and they're evaluated at the point they appear, which is both faster and clearer.


In [2]:
name = 'Ada'
count = 3
price = 19.999

old_style = '%s bought %d items for $%.2f' % (name, count, price)
format_style = '{} bought {} items for ${:.2f}'.format(name, count, price)
fstring_style = f'{name} bought {count} items for ${price:.2f}'

print(old_style)
print(format_style)
print(fstring_style)

# F-strings also allow arbitrary expressions inline, not just variable names
items = ['a', 'b', 'c']
print(f'There are {len(items)} items, doubled: {len(items) * 2}')


Ada bought 3 items for $20.00
Ada bought 3 items for $20.00
Ada bought 3 items for $20.00
There are 3 items, doubled: 6


### Item 12 — Understand the Difference Between `repr` and `str` when Printing Objects

`str(obj)` produces a human-readable display (what `print()` uses by default); `repr(obj)` produces an unambiguous, ideally Python-executable representation useful for debugging (what you see when you type a variable name alone in a REPL, or inspect it in a list). For a string, `str('5')` is `5` but `repr('5')` is `'5'` — the quotes matter when you're trying to tell a string `'5'` apart from an integer `5` while debugging.


In [3]:
value_str = '5'
value_int = 5

print(str(value_str), str(value_int))     # both just print "5" -- looks identical!
print(repr(value_str), repr(value_int))   # repr shows the quotes -- now the difference is obvious

class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __repr__(self):
        return f'Point(x={self.x!r}, y={self.y!r})'   # unambiguous, could recreate the object

p = Point(1, 2)
print(p)          # uses __repr__ since Point has no __str__ of its own
print(f'{p!r}')   # the !r conversion flag forces repr() inside an f-string


5 5
'5' 5
Point(x=1, y=2)
Point(x=1, y=2)


### Item 13 — Prefer Explicit String Concatenation over Implicit, Especially in Lists

Python silently concatenates adjacent string literals (`'a' 'b'` becomes `'ab'`) with no operator at all — convenient for wrapping a long string across lines, but dangerous inside a list or function call, where a missing comma between two string literals silently merges them into one item instead of raising an error.


In [4]:
# Implicit concatenation across lines: intentional and fine
message = ('This is a long message '
           'split across multiple lines '
           'using implicit concatenation.')
print(message)

# The dangerous case: a missing comma in a list silently merges two items into one
items_with_bug = [
    'apple',
    'banana'
    'cherry',   # <-- missing comma above merged 'banana' and 'cherry' into one string!
]
print(items_with_bug)
print(len(items_with_bug))   # 2 items, not 3 -- the bug is easy to miss visually

# Explicit concatenation with + makes the intent (and the comma requirement) obvious
items_explicit = [
    'apple',
    'banana' + 'cherry',   # clearly one deliberate combined item, not a typo
]
print(items_explicit)


This is a long message split across multiple lines using implicit concatenation.
['apple', 'bananacherry']
2
['apple', 'bananacherry']


### Item 14 — Know How to Slice Sequences

`seq[start:stop]` returns a new sequence from index `start` up to (not including) `stop`. Either bound can be omitted (`seq[:3]`, `seq[3:]`) to mean "from the beginning" / "to the end." Negative indices count from the end (`seq[-2:]` is the last two items). Slicing never raises an `IndexError` even if the bounds go past the sequence's length — it just clips to what's available, unlike direct indexing.


In [5]:
letters = ['a', 'b', 'c', 'd', 'e', 'f']

print(letters[2:5])     # index 2 up to (not including) 5
print(letters[:3])       # from the start
print(letters[3:])       # to the end
print(letters[-2:])      # last two items
print(letters[1:20])     # out-of-range stop -- clips silently, no error

try:
    letters[20]           # direct indexing DOES raise when out of range
except IndexError as e:
    print('Direct indexing raised:', e)


['c', 'd', 'e']
['a', 'b', 'c']
['d', 'e', 'f']
['e', 'f']
['b', 'c', 'd', 'e', 'f']
Direct indexing raised: list index out of range


### Item 15 — Avoid Striding and Slicing in a Single Expression

`seq[start:stop:step]` adds a third component, the stride — `seq[::2]` takes every second item. Combining a stride with non-trivial start/stop bounds in one expression (`seq[2:8:2]`) gets confusing fast, especially with negative strides (which also reverse direction). The book's advice: do the slice and the stride as two separate, named steps, or use `itertools.islice` for anything beyond the simplest `[::2]`/`[::-1]` cases.


In [6]:
values = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

print(values[::2])       # simple, clear stride: every 2nd item
print(values[::-1])      # simple, clear: reversed

# Confusing: stride AND bounds combined in one expression
confusing = values[2:8:2]
print(confusing)   # what does this even select? Hard to tell at a glance.

# Clearer: split into two separate, named steps
sliced = values[2:8]      # first: the bounds
strided = sliced[::2]     # then: the stride
print(strided)            # same result, much easier to read


[0, 2, 4, 6, 8]
[9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
[2, 4, 6]
[2, 4, 6]


### Item 16 — Prefer Catch-All Unpacking over Slicing

When you need "the first item and everything else" (or "everything except the last item"), avoid manual slicing (`rest = items[1:]`) in favor of a starred catch-all in an unpacking assignment: `first, *rest = items`. It works regardless of the sequence's length, is less error-prone than hand-counting indices, and reads as intent ("first, and the rest") rather than an arithmetic offset.


In [7]:
scores = [100, 87, 92, 65, 78]

# Slicing approach: works, but is an arithmetic offset you have to double check
first_sliced = scores[0]
rest_sliced = scores[1:]
print(first_sliced, rest_sliced)

# Catch-all unpacking: same result, reads as plain intent
first, *rest = scores
print(first, rest)

# Works from either end, and in the middle too
*most, last = scores
print(most, last)

first, *middle, last = scores
print(first, middle, last)


100 [87, 92, 65, 78]
100 [87, 92, 65, 78]
[100, 87, 92, 65] 78
100 [87, 92, 65] 78
